# 加载模型

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("正在加载模型（首次运行会下载约 500MB）...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载模型（首次运行会下载约 500MB）...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1196.97it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# 查看模型内部向量表

In [2]:
embedding_table = model.transformer.wte.weight
print(embedding_table.shape)  # torch.Size([50257, 768])

torch.Size([50257, 768])


In [3]:
token_id = tokenizer.encode("France")[0]
vector = embedding_table[token_id]
print(vector.shape)                                # torch.Size([768])
print(vector[:16])                                  # 前 8 维: [-0.133, 0.098, 0.230, ...]

torch.Size([768])
tensor([-0.1329,  0.0977,  0.2297, -0.1049, -0.0112,  0.0470, -0.2165, -0.0211,
        -0.1277,  0.0466, -0.0395, -0.0616, -0.0606,  0.0109,  0.1844, -0.0117],
       grad_fn=<SliceBackward0>)


# 语义相似度

In [4]:
import torch

def cosine_similarity(v1, v2):
    return torch.dot(v1, v2) / (v1.norm() * v2.norm())

france = embedding_table[tokenizer.encode("France")[0]]
paris  = embedding_table[tokenizer.encode("Paris")[0]]
cat    = embedding_table[tokenizer.encode("cat")[0]]

sim1 = cosine_similarity(france, paris)  # ≈ 0.65  较高，有关系
sim2 = cosine_similarity(france, cat)    # ≈ 0.24  低，不相关
print(f"sim1 = {sim1}")
print(f"sim2 = {sim2}")

sim1 = 0.6504826545715332
sim2 = 0.24161531031131744


In [7]:
king = embedding_table[tokenizer.encode("king")[0]]
man = embedding_table[tokenizer.encode("man")[0]]
woman = embedding_table[tokenizer.encode("woman")[0]]
queen = embedding_table[tokenizer.encode("queen")[0]]
france = embedding_table[tokenizer.encode("France")[0]]
germany = embedding_table[tokenizer.encode("Germany")[0]]
paris = embedding_table[tokenizer.encode("Paris")[0]]
berlin = embedding_table[tokenizer.encode("Berlin")[0]]


print(f"king & queen = {cosine_similarity(king, queen)}")
change = king - man + woman
print(f"change & queen = {cosine_similarity(change, queen)}")

print(f"paris & berlin = {cosine_similarity(paris, berlin)}")
change = paris - france + germany
print(f"change & berlin = {cosine_similarity(change, berlin)}")

king & queen = 0.2666102647781372
change & queen = 0.2777973413467407
paris & berlin = 0.45116427540779114
change & berlin = 0.4842508137226105


# 比较任意词对的相似度

In [10]:
# ==================== 取出 Embedding 表 ====================

embedding_table = model.transformer.wte.weight.detach()
print(f"\nEmbedding 表的形状: {embedding_table.shape}")
print(f"  行数（词表大小）: {embedding_table.shape[0]}")
print(f"  列数（向量维度）: {embedding_table.shape[1]}")
print(f"  总参数量: {embedding_table.shape[0] * embedding_table.shape[1]:,}")

def get_vector(word):
    """获取一个词的 Embedding 向量，自动处理带/不带空格的变体"""
    for variant in [word, " " + word, word.lower(), " " + word.lower(),
                    word.capitalize(), " " + word.capitalize()]:
        ids = tokenizer.encode(variant)
        if len(ids) == 1:
            return embedding_table[ids[0]], ids[0], variant.strip()
    ids = tokenizer.encode(word)
    return embedding_table[ids[0]], ids[0], tokenizer.decode([ids[0]]).strip()

words = ["king", "man", "woman", "queen"]

# ==================== 展示向量 ====================

print(f"\n各 token 的向量（前 8 维）：")
print("-" * 70)

vectors = {}
for word in words:
    vec, tid, display = get_vector(word)
    vectors[word] = vec
    preview = [f"{v:.3f}" for v in vec[:8].tolist()]
    print(f"  '{display}' (ID {tid}): [{', '.join(preview)}, …]")

# ==================== 相似度矩阵 ====================

print(f"\n\n向量相似度（余弦相似度）：")
print("=" * 60)

computed_pairs = set()
results = []

for w1 in words:
    for w2 in words:
        if w1 >= w2:
            continue
        key = tuple(sorted([w1, w2]))
        if key in computed_pairs:
            continue
        computed_pairs.add(key)

        if w1 in vectors and w2 in vectors:
            sim = cosine_similarity(vectors[w1], vectors[w2]).item()
            results.append((w1, w2, sim))

# 按相似度降序排列
results.sort(key=lambda x: -x[2])

for w1, w2, sim in results:
    bar = "█" * int(max(0, sim) * 30)
    print(f"  '{w1}' vs '{w2}': {sim:>7.3f}  {bar}")

# ==================== 向量统计 ====================

print(f"\n\n向量统计信息：")
print("-" * 40)

for word in words[:5]:
    if word in vectors:
        vec = vectors[word]
        print(f"  '{word}': 范数={vec.norm().item():.3f}, "
              f"均值={vec.mean().item():.4f}, "
              f"标准差={vec.std().item():.4f}")


Embedding 表的形状: torch.Size([50257, 768])
  行数（词表大小）: 50257
  列数（向量维度）: 768
  总参数量: 38,597,376

各 token 的向量（前 8 维）：
----------------------------------------------------------------------
  'king' (ID 3364): [0.109, -0.395, 0.059, -0.102, -0.279, 0.260, -0.312, -0.088, …]
  'man' (ID 805): [-0.134, -0.138, 0.090, 0.065, -0.005, -0.011, -0.273, -0.018, …]
  'woman' (ID 8580): [-0.096, -0.077, 0.086, 0.251, -0.093, 0.065, -0.298, -0.116, …]
  'queen' (ID 16599): [-0.020, -0.290, -0.032, -0.059, 0.052, -0.083, -0.337, 0.015, …]


向量相似度（余弦相似度）：
  'man' vs 'woman':   0.586  █████████████████
  'queen' vs 'woman':   0.385  ███████████
  'king' vs 'woman':   0.294  ████████
  'king' vs 'man':   0.291  ████████
  'king' vs 'queen':   0.290  ████████
  'man' vs 'queen':   0.217  ██████


向量统计信息：
----------------------------------------
  'king': 范数=4.152, 均值=0.0016, 标准差=0.1499
  'man': 范数=3.564, 均值=-0.0009, 标准差=0.1287
  'woman': 范数=4.113, 均值=0.0021, 标准差=0.1485
  'queen': 范数=3.785, 均值=-0.0000, 标准

# 降维聚类

## 构建可视化的词

In [5]:
embedding_table = model.transformer.wte.weight.detach()

word_groups = {
    "国家": ["France", "Germany", "Japan", "China", "Italy", "Spain", "Russia", "Brazil", "India"],
    "城市": ["Paris", "Berlin", "Tokyo", "Beijing", "Rome", "Madrid", "Moscow", "London", "Delhi"],
    "动物": ["cat", "dog", "fish", "bird", "horse", "tiger", "lion", "bear", "wolf"],
    "颜色": ["red", "blue", "green", "yellow", "black", "white", "purple", "orange"],
    "数字": ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"],
    "编程": ["Python", "Java", "code", "function", "variable", "class", "data", "array"],
    "食物": ["apple", "bread", "rice", "cheese", "pizza", "chicken", "chocolate", "coffee"],
}

all_words = []
all_vectors = []
all_groups = []

for group, words in word_groups.items():
    for word in words:
        for variant in [" " + word, word, word.lower(), " " + word.lower()]:
            ids = tokenizer.encode(variant)
            if len(ids) == 1:
                vec = embedding_table[ids[0]].numpy()
                all_words.append(word)
                all_vectors.append(vec)
                all_groups.append(group)
                break

print(f"收集了 {len(all_words)} 个词的向量（来自 {len(word_groups)} 个语义组）")
print(f"all_words: {all_words}")
print(f"all_vectors: {all_vectors}")
print(f"all_groups: {all_groups}")

收集了 61 个词的向量（来自 7 个语义组）
all_words: ['France', 'Germany', 'Japan', 'China', 'Italy', 'Spain', 'Russia', 'Brazil', 'India', 'Paris', 'Berlin', 'Tokyo', 'Beijing', 'Rome', 'Madrid', 'Moscow', 'London', 'Delhi', 'cat', 'dog', 'fish', 'bird', 'horse', 'tiger', 'lion', 'bear', 'wolf', 'red', 'blue', 'green', 'yellow', 'black', 'white', 'purple', 'orange', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'Python', 'Java', 'code', 'function', 'variable', 'class', 'data', 'array', 'apple', 'bread', 'rice', 'cheese', 'pizza', 'chicken', 'chocolate', 'coffee']
all_vectors: [array([-6.94251284e-02,  4.89273183e-02,  1.62446752e-01, -7.34931082e-02,
        4.06064503e-02,  6.07754476e-02, -2.26280183e-01,  3.28108959e-04,
       -6.83388934e-02,  7.90781751e-02, -1.46123366e-02, -8.59463811e-02,
        1.04389727e-01,  5.62340170e-02, -2.19830647e-02,  6.49288595e-02,
        5.05471341e-02,  9.83921513e-02,  1.69831693e-01,  9.44533795e-02,
        2.00185981e-02, -

## PCA降维

In [7]:
import numpy as np

vectors_matrix = np.array(all_vectors)
mean = vectors_matrix.mean(axis=0)
centered = vectors_matrix - mean

cov = centered.T @ centered / len(centered)
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# 取最大的两个特征向量
pc1 = eigenvectors[:, -1]
pc2 = eigenvectors[:, -2]

x = centered @ pc1
y = centered @ pc2

# 解释的方差比例
total_var = eigenvalues.sum()
explained_1 = eigenvalues[-1] / total_var * 100
explained_2 = eigenvalues[-2] / total_var * 100
print(f"PCA: PC1 解释 {explained_1:.1f}% 方差, PC2 解释 {explained_2:.1f}% 方差")

PCA: PC1 解释 12.1% 方差, PC2 解释 7.4% 方差


# 多义词

## 函数定义

In [11]:
import torch

def get_all_layer_vectors(text, target_word):
    """获取目标词在每一层 Transformer 输出的向量"""
    input_ids = tokenizer.encode(text, return_tensors="pt")
    print(input_ids)
    tokens = [tokenizer.decode([id]) for id in input_ids[0]]
    print(tokens)

    # 找到目标词的位置
    target_pos = None
    for i, t in enumerate(tokens):
        if target_word.lower() in t.strip().lower():
            target_pos = i
            break

    if target_pos is None:
        return None, tokens, None

    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True)
        print(outputs.hidden_states)

    # hidden_states[0] = Embedding 层, [1]-[12] = Transformer 各层
    layer_vectors = [hs[0, target_pos, :].detach() for hs in outputs.hidden_states]

    return layer_vectors, tokens, target_pos

def cosine_sim(v1, v2):
    return torch.dot(v1, v2) / (v1.norm() * v2.norm())

## 一词多义

In [14]:
print("\n" + "=" * 65)
print("实验 1: 'bank' 在不同语境中的向量变化")
print("=" * 65)

sentences = [
    ("I deposited money at the bank today", "bank", "银行"),
    ("The river bank was covered with flowers", "bank", "河岸"),
    ("The bank approved my mortgage application", "bank", "银行"),
    ("We had a picnic on the bank of the lake", "bank", "河岸"),
]
results = []
for text, word, meaning in sentences:
    vectors, tokens, pos = get_all_layer_vectors(text, word)
    if vectors:
        results.append({"text": text, "meaning": meaning, "vectors": vectors})
        print(f"\n  [{meaning}] '{text}'")
        print(f"  tokens: {tokens}, 目标位置: {pos}")

if len(results) >= 4:
    print(f"\n\n相似度对比：Embedding 层 vs 最后一层")
    print("-" * 65)

    comparisons = [
        (0, 2, "银行 vs 银行"),
        (1, 3, "河岸 vs 河岸"),
        (0, 1, "银行 vs 河岸"),
        (0, 3, "银行 vs 河岸"),
    ]

    print(f"\n  {'对比':<25} {'Embedding':>10} {'最后一层':>10} {'变化':>10}")
    print("  " + "-" * 60)

    for i, j, label in comparisons:
        sim_e = cosine_sim(results[i]["vectors"][0], results[j]["vectors"][0]).item()
        sim_f = cosine_sim(results[i]["vectors"][-1], results[j]["vectors"][-1]).item()
        diff = sim_f - sim_e
        arrow = "↑" if diff > 0.001 else ("↓" if diff < -0.001 else "→")
        print(f"  {label:<25} {sim_e:>10.3f} {sim_f:>10.3f}   {arrow} {diff:+.3f}")
    
    # 逐层追踪
    print(f"\n\n逐层追踪：银行 vs 河岸 的 'bank' 相似度")
    print("=" * 50)

    print(f"\n  {'层':<14} {'相似度':>8}  可视化")
    print("  " + "-" * 45)

    for layer in range(len(results[0]["vectors"])):
        sim = cosine_sim(results[0]["vectors"][layer], results[1]["vectors"][layer]).item()
        bar = "█" * int(max(0, sim) * 30)
        name = "Embedding" if layer == 0 else f"Layer {layer:>2}"
        print(f"  {name:<14} {sim:>8.3f}  {bar}")

    print(f"\n  Embedding 层相似度 = 1.000（同一个 token，同一行向量）")
    print(f"  最后一层相似度 < 1.000（上下文不同，语义不同）")
    print("  → Transformer 逐层将'一词一义'的固定向量")
    print("    变成了'一词多义'的上下文感知向量")


实验 1: 'bank' 在不同语境中的向量变化
tensor([[   40, 27163,  1637,   379,   262,  3331,  1909]])
['I', ' deposited', ' money', ' at', ' the', ' bank', ' today']
(tensor([[[ 0.1286, -0.2933,  0.1470,  ...,  0.0599, -0.0342, -0.0586],
         [ 0.1443, -0.2173, -0.0392,  ...,  0.2290,  0.1136,  0.0674],
         [ 0.1244, -0.2318,  0.1877,  ...,  0.1141, -0.0357, -0.1701],
         ...,
         [-0.0317, -0.0201,  0.1691,  ..., -0.0393,  0.0768, -0.0541],
         [ 0.0553,  0.0480,  0.2687,  ...,  0.0743, -0.0453, -0.1384],
         [ 0.0684,  0.0166,  0.1755,  ...,  0.2105, -0.0366, -0.0664]]]), tensor([[[ 0.0532, -0.7611,  0.4959,  ..., -1.2261, -0.0817,  0.6238],
         [ 0.1484, -0.5578, -0.8298,  ..., -0.8183,  1.1528,  0.6540],
         [ 0.1650, -1.7438, -0.6737,  ...,  0.9279, -0.1797, -0.5402],
         ...,
         [-0.8319,  0.4140, -0.2910,  ...,  0.2278,  0.3963, -0.3942],
         [ 0.0693,  0.2780, -0.8461,  ...,  0.1867, -0.4156, -1.7049],
         [ 0.2996, -0.0499, -0.7526, 

# 位置编码

In [8]:
position_table = model.transformer.wpe.weight
print(position_table.shape)  # torch.Size([1024, 768])
# 1024 个位置，每个位置 768 维 → GPT-2 最多处理 1024 个 token

torch.Size([1024, 768])
